In [1]:
from graph_build import create_dummy_knowledge_graph
from graph_retriver import GraphRetriver


c:\Users\migue\Documents\Code_projects\Python\envs\AI_general_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 646.05it/s, Materializing param=pooler.dense.weight]                             


In [4]:
import re
import os
triplets = []
with open('Outputs/raw_triplets.txt', 'r') as f:
    text = f.read()
pattern = r"['\"](.*?)['\"],\s*['\"](.*?)['\"],\s*['\"](.*?)['\"]"
result = re.findall(pattern, text)

    


In [6]:
type(result[1])

tuple

In [ ]:
from datasets import load_dataset

# Load a small slice of HotpotQA (multi-hop reasoning)
dataset = load_dataset("hotpot_qa", "distractor", split="train", streaming=True)
sample = next(iter(dataset))

print(f"Question: {sample['question']}")
print(f"Context: {sample['context']}")


c:\Users\migue\Documents\Code_projects\Python\envs\AI_general_env\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\migue\.cache\huggingface\hub\datasets--hotpot_qa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Question: Which magazine was started first Arthur's Magazine or First for Women?
Context: {'title': ['Radio City (Indian radio station)', 'History of Albanian football', 'Echosmith', "Women's colleges in the Southern United States", 'First Arthur County Courthouse and Jail', "Arthur's Magazine", '2014–15 Ukrainian Hockey Championship', 'First for Women', 'Freeway Complex Fire', 'William Rast'], 'sentences': [["Radio City is India's first private FM radio station and was started on 3 July 2001.", ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).', ' It plays Hindi, English and regional songs.', ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.', ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, a

In [5]:
import time 
from smolagents import InferenceClientModel
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN")
model_for_extraction = InferenceClientModel(model_id = 'Qwen/Qwen2.5-Coder-7B-Instruct')
model_for_answer = InferenceClientModel(model_id = 'Qwen/Qwen2.5-Coder-7B-Instruct')

In [8]:
from collections import deque
kg = create_dummy_knowledge_graph()
graph_retrive = GraphRetriver(kg)
query = "What was the relation between adolf hitler and einstein?"
retrived_triplets = graph_retrive.retrive_triplets_from_knowledgegraph(query)
fitlerd_triplets = graph_retrive.filter_relevant_triplets(query,retrived_triplets)
print('All triplets:')
for triplet in retrived_triplets: print(triplet)
print('\nFilterd triplets')
for triplet in fitlerd_triplets: print(triplet)


All triplets:
('Albert Einstein', 'born on', '14 March 1879')
('Albert Einstein', 'died on', '18 April 1955')
('Albert Einstein', 'nationality', 'German-born')
('Albert Einstein', 'known for', 'developing the theory of relativity')
('Albert Einstein', 'contributed to', 'quantum theory')
('Albert Einstein', 'mass-energy equivalence formula', 'E = mc^2')
('Albert Einstein', 'received Nobel Prize in Physics in', '1921')
('Albert Einstein', 'reason for Nobel Prize', 'his services to theoretical physics, and especially for his discovery of the law of the photoelectric effect')
('Albert Einstein', 'moved to', 'Switzerland in 1895')
('Albert Einstein', 'forsook German citizenship', 'in 1896')
('Albert Einstein', 'enrolled in', 'mathematics and physics teaching diploma program')
('Albert Einstein', 'program location', 'Swiss federal polytechnic school in Zurich')
('Albert Einstein', 'graduated from', 'University of Zurich in 1900')
('Albert Einstein', 'acquired Swiss citizenship', 'in 1901')
(

# Using Smolagent

In [102]:
from smolagents import LiteLLMModel, ChatMessage
#Define the model to answer the query with the context:
# Initialize the local Ollama model
model = LiteLLMModel(
    model_id="ollama_chat/qwen2.5:1.5b-Instruct", 
    api_base="http://localhost:11434", # Default Ollama port
    num_ctx=8192 # Context window
)
messages = [
            ChatMessage(role="system", content=[{"type": "text", "text": model_instructions+context}]),
            ChatMessage(role="user", content=[{"type": "text", "text": query}])
        ]
answer = model.generate(messages=messages)
print(answer.content)

Hitler was a Nazi who persecuted Jews, including Albert Einstein.


# Using Ollama model

In [103]:
import ollama
stream = ollama.chat(
    model='qwen2.5:1.5b-Instruct',
    messages=[{'role': 'system', 'content': model_instructions + context},
              {'role': 'user', 'content': query}],
    stream=True,
)
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

Not enough information.

# Using HF inference model

In [105]:
from dotenv import load_dotenv
from smolagents import InferenceClientModel
import os
load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN")
model_for_extraction = InferenceClientModel(model_id = 'Qwen/Qwen2.5-Coder-7B-Instruct') 
messages=[{'role': 'system', 'content': model_instructions + context},
                        {'role': 'user', 'content': query}]
model_for_extraction(messages).content

'Hitler persecuted Jews.  \nAlbert Einstein developed the Theory of Relativity.  \nAlbert Einstein was Jewish.'